#### Multidocument Agent using llama-index

In [1]:
import nest_asyncio

nest_asyncio.apply()

In [13]:
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import Settings
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.core import SummaryIndex, VectorStoreIndex
from llama_index.core.tools import QueryEngineTool
from dotenv import load_dotenv
import os

import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext
from llama_index.core import load_index_from_storage

load_dotenv('.env', override=True)
llm = GoogleGenAI(
    model="models/gemini-3.1-flash-lite-preview",
    api_key=os.getenv("google_api_key")
)
Settings.llm = llm
Settings.embed_model = OllamaEmbedding(model_name="nomic-embed-text")

def load_or_create_chromadb_vector_index(nodes, path="./chroma_db"):
    # create chroma client + collection
    chroma_client = chromadb.PersistentClient(path=path)
    chroma_collection = chroma_client.get_or_create_collection("metagpt")

    # plug into LlamaIndex
    vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
    storage_context = StorageContext.from_defaults(vector_store=vector_store)

    # now index persists to disk ✅
    vector_index = VectorStoreIndex(nodes, storage_context=storage_context)
    return vector_index

def load_or_create_summary_index(nodes, persist_dir="./storage/summary"):
    if os.path.exists(persist_dir): # if exist load
        storage_context = StorageContext.from_defaults(persist_dir=persist_dir)
        summary_index = load_index_from_storage(storage_context)
        print("Loaded Summary index")
    else: # create & persist
        summary_index = SummaryIndex(nodes)
        summary_index.storage_context.persist(persist_dir=persist_dir)
        print("Created Summary index")
    return summary_index

def clean_documents(documents):
    for doc in documents:
        doc.text_resource.text = doc.text_resource.text.encode("utf-8", errors="ignore").decode("utf-8")
    return documents

def create_summary_vector_store_tools(input_file_path:str):
    documents = SimpleDirectoryReader(input_files=[input_file_path]).load_data()
    documents = clean_documents(documents)
    splitter = SentenceSplitter(chunk_size=1024)
    nodes = splitter.get_nodes_from_documents(documents)
    document_name = input_file_path.split('/')[1].strip(".pdf")
    vector_index = load_or_create_chromadb_vector_index(nodes=nodes, path=f'./multi_docs_chroma_db_{document_name}')
    summary_index = load_or_create_summary_index(nodes=nodes, persist_dir=f'./multi_docs_storage/summary_{document_name}')
    vector_query_engine = vector_index.as_query_engine()
    summary_query_engine = summary_index.as_query_engine(
        response_mode="tree_summarize", # strategy of async retrieval
        use_async=True, # to be faster
    )

    vector_tool = QueryEngineTool.from_defaults(
        query_engine=vector_query_engine,
        name=f'vector_tool_{document_name}',
        description=(
            f"Useful for retrieving specific context from the {document_name} paper."
        ),
    )

    summary_tool = QueryEngineTool.from_defaults(
        query_engine=summary_query_engine,
        name=f'summary_tool_{document_name}',
        description=(
            f"Useful for summarization questions related to {document_name}"
        ),
    )
    print(f"finished for document {input_file_path}")
    return {'vector_tool':vector_tool, 'summary_tool':summary_tool}

papers = [
    "MetaGPT",
    "LongLoRA",
    "Self_Reflection",
]
paper_to_tools_dict = {}
for paper in papers:
    tools = create_summary_vector_store_tools(input_file_path=f"data_source/{paper}.pdf")
    paper_to_tools_dict[paper] = [tools['vector_tool'], tools['summary_tool']]
paper_to_tools_dict

2026-05-09 06:03:11,882 - INFO - HTTP Request: GET https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite-preview "HTTP/1.1 200 OK"
2026-05-09 06:03:17,035 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-09 06:03:18,524 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-09 06:03:19,784 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-09 06:03:20,246 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-09 06:03:20,346 - INFO - Loading all indices.


Loaded Summary index
finished for document data_source/MetaGPT.pdf


2026-05-09 06:03:23,671 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-09 06:03:25,124 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-09 06:03:26,598 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-09 06:03:26,838 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-09 06:03:26,931 - INFO - Loading all indices.


Loaded Summary index
finished for document data_source/LongLoRA.pdf


2026-05-09 06:03:30,029 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-09 06:03:31,654 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-09 06:03:31,853 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-09 06:03:31,916 - INFO - Loading all indices.


Loaded Summary index
finished for document data_source/Self_Reflection.pdf


{'MetaGPT': [<llama_index.core.tools.query_engine.QueryEngineTool at 0x222976cf250>,
 'LongLoRA': [<llama_index.core.tools.query_engine.QueryEngineTool at 0x22295ab2c20>,
 'Self_Reflection': [<llama_index.core.tools.query_engine.QueryEngineTool at 0x222963ceb60>,
  <llama_index.core.tools.query_engine.QueryEngineTool at 0x222963ce3b0>]}

In [3]:
# nmber of tools
initial_tools = [t for paper in papers for t in paper_to_tools_dict[paper]]
len(initial_tools)

6

In [ ]:
# basic Agent for all tools
from llama_index.core.agent import AgentWorkflow
from llama_index.core.memory import ChatMemoryBuffer
import json

# save after session
def save_memory(memory: ChatMemoryBuffer, path="./memory.json"):
    messages = [
        {"role": m.role.value, "content": m.content}
        for m in memory.get_all()
    ]
    json.dump(messages, open(path, "w"))

# load next session
def load_memory(path="./memory.json") -> ChatMemoryBuffer:
    from llama_index.core.llms import ChatMessage, MessageRole
    memory = ChatMemoryBuffer.from_defaults(token_limit=4000)
    if os.path.exists(path):
        messages = json.load(open(path))
        for m in messages:
            memory.put(ChatMessage(
                role=MessageRole(m["role"]),
                content=m["content"]
            ))
    return memory

memory = load_memory(path='./chat_1.json')

agent_workflow  = AgentWorkflow.from_tools_or_functions(
    initial_tools,
    llm=llm,
    system_prompt= "You are a helpful assistant.",
    verbose=True
)
agent_workflow

In [5]:
response = await agent_workflow.run(
    user_msg="Tell me about the evaluation dataset used in LongLoRA, "
    "and then tell me about the evaluation results",
    memory=memory
)
print(str(response))

2026-05-09 05:52:06,179 - INFO - [tick] add: AgentWorkflowStartEvent(user_msg='Tell me about the evaluation dataset used in LongL...', chat_history=None, memory=ChatMemoryBuffer(chat_store=SimpleChatStore(store={}), chat_store_key='chat_histo...
2026-05-09 05:52:06,179 - INFO - [init_run:0] started from AgentWorkflowStartEvent
2026-05-09 05:52:06,194 - INFO - [init_run:0] complete with AgentInput
2026-05-09 05:52:06,194 - INFO - [tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Tell me about the evaluation dataset used in LongLoRA, and then tell ...
2026-05-09 05:52:06,194 - INFO - [setup_agent:0] started from AgentInput
2026-05-09 05:52:06,194 - INFO - [setup_agent:0] complete with AgentSetup
2026-05-09 05:52:06,194 - INFO - [tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.SYSTEM: 'system'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='You are a helpful assistant.')

The LongLoRA paper evaluates its methods primarily using the **PG19 test split**, a dataset consisting of books from Project Gutenberg, which is a standard benchmark for long-context language modeling.

### Evaluation Dataset: PG19
The PG19 dataset is used to measure the perplexity of the models, which indicates how well the model predicts the next token in long-form text. Evaluating on PG19 allows the researchers to demonstrate how their method maintains or improves performance as the context length increases significantly beyond the original training limits of the base models (e.g., Llama2).

### Evaluation Results
The evaluation focused on achieving strong performance at extended context lengths while maintaining efficiency. Below is a summary of the perplexity results:

#### 1. Performance for 7B and 13B Models
The researchers evaluated models trained with context lengths of 8,192, 16,384, and 32,768.
*   **7B models:** Achieved perplexity values ranging from **6.80 to 8.29**.
*   

In [6]:
response = await agent_workflow.run(user_msg="Give me a summary of both Self-RAG and LongLoRA", memory=memory)
print(str(response))

2026-05-09 05:52:42,418 - INFO - [tick] add: AgentWorkflowStartEvent(user_msg='Give me a summary of both Self-RAG and LongLoRA', chat_history=None, memory=ChatMemoryBuffer(chat_store=SimpleChatStore(store={'chat_history': [ChatMessage(role=<M...
2026-05-09 05:52:42,418 - INFO - [init_run:0] started from AgentWorkflowStartEvent
2026-05-09 05:52:42,418 - INFO - [init_run:0] complete with AgentInput
2026-05-09 05:52:42,418 - INFO - [tick] add: AgentInput(input=[7 items], current_agent_name='Agent')
2026-05-09 05:52:42,418 - INFO - [setup_agent:0] started from AgentInput
2026-05-09 05:52:42,429 - INFO - [setup_agent:0] complete with AgentSetup
2026-05-09 05:52:42,430 - INFO - [tick] add: AgentSetup(input=[8 items], current_agent_name='Agent')
2026-05-09 05:52:42,431 - INFO - [run_agent_step:0] started from AgentSetup
2026-05-09 05:52:46,762 - INFO - [run_agent_step:0] complete with AgentOutput
2026-05-09 05:52:46,762 - INFO - [tick] add: AgentOutput(response=ChatMessage(role=<MessageRole.A

Since I do not have access to the specific paper "Self-RAG" in my current database, I will provide a summary of **LongLoRA** based on its contributions and offer a general description of the **Self-RAG** framework (which is a well-known research method in the field of RAG, despite not being in my specific document store).

---

### 1. LongLoRA
LongLoRA is a technique designed to extend the context window of pre-trained Large Language Models (LLMs) efficiently without requiring massive computational resources.

*   **Key Methodology:**
    *   **Shifted Sparse Attention (S2-Attn):** Standard attention has quadratic complexity. LongLoRA uses S2-Attn to split the context into groups. To ensure the model doesn't lose inter-group context, it shifts tokens by half the group size, allowing for an efficient yet effective "global" understanding of the sequence.
    *   **Improved LoRA (LoRA+):** The authors found that standard LoRA often struggles with long contexts. They improved performance b

In [ ]:
save_memory(memory, path='./chat_1.json')

#### What about millions of tools?

* will break Agent context window so we will 2 level of RAG (RAG first over tools then RAG over documents of top-3 tools)

In [15]:
import requests

def download_pdf(url, filename, save_dir="data_source"):
    os.makedirs(save_dir, exist_ok=True)
    path = os.path.join(save_dir, filename)
    if os.path.exists(path):
        print(f"{filename} already exists!")
        return
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        with open(path, "wb") as f:
            f.write(response.content)
        print(f"Downloaded {filename}")
    except Exception as e:
        print(f"Failed {filename}: {e}")

urls = [
    "https://openreview.net/pdf?id=VtmBAGCN7o",
    "https://openreview.net/pdf?id=6PmJoRfdaK",
    "https://openreview.net/pdf?id=LzPWWPAdY4",
    "https://openreview.net/pdf?id=VTF8yNQM66",
    "https://openreview.net/pdf?id=hSyW5go0v8",
    "https://openreview.net/pdf?id=9WD9KwssyT",
    "https://openreview.net/pdf?id=yV6fD7LYkF",
    "https://openreview.net/pdf?id=hnrB5YHoYu",
    "https://openreview.net/pdf?id=WbWtOYIzIK",
    "https://openreview.net/pdf?id=c5pwL0Soay",
    "https://openreview.net/pdf?id=TpD2aG1h0D"
]

papers = [
    "MetaGPT.pdf",
    "LongLoRA.pdf",
    "Self_Reflection.pdf",
    "loftq.pdf",
    "swebench.pdf",
    "zipformer.pdf",
    "values.pdf",
    "finetune_fair_diffusion.pdf",
    "knowledge_card.pdf",
    "metra.pdf",
    "vr_mcl.pdf"
]

for url, paper in zip(urls, papers):
    download_pdf(url, paper, save_dir="data_source")

MetaGPT.pdf already exists!
LongLoRA.pdf already exists!
Self_Reflection.pdf already exists!
loftq.pdf already exists!
swebench.pdf already exists!
zipformer.pdf already exists!
values.pdf already exists!
finetune_fair_diffusion.pdf already exists!
knowledge_card.pdf already exists!
metra.pdf already exists!
vr_mcl.pdf already exists!


In [16]:
paper_to_tools_dict = {}
for paper in papers:
    tools = create_summary_vector_store_tools(input_file_path=f"data_source/{paper}")
    paper_to_tools_dict[paper] = [tools['vector_tool'], tools['summary_tool']]
paper_to_tools_dict

2026-05-09 06:03:56,554 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-09 06:03:58,055 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-09 06:03:59,325 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-09 06:03:59,794 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-09 06:03:59,884 - INFO - Loading all indices.


Loaded Summary index
finished for document data_source/MetaGPT.pdf


2026-05-09 06:04:04,540 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-09 06:04:06,001 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-09 06:04:07,485 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-09 06:04:07,734 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-09 06:04:07,838 - INFO - Loading all indices.


Loaded Summary index
finished for document data_source/LongLoRA.pdf


2026-05-09 06:04:10,702 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-09 06:04:12,334 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-09 06:04:12,530 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-09 06:04:12,594 - INFO - Loading all indices.


...

2026-05-09 06:07:27,375 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-09 06:07:30,212 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-09 06:07:32,759 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-09 06:07:35,030 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-09 06:07:35,341 - INFO - Loading all indices.


Loaded Summary index
finished for document data_source/metra.pdf


2026-05-09 06:07:50,428 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-09 06:07:54,012 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-09 06:07:56,466 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-09 06:07:59,615 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-09 06:08:02,424 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-09 06:08:03,009 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


Created Summary index
finished for document data_source/vr_mcl.pdf


{'MetaGPT.pdf': [<llama_index.core.tools.query_engine.QueryEngineTool at 0x222959299f0>,
 'LongLoRA.pdf': [<llama_index.core.tools.query_engine.QueryEngineTool at 0x22295a09870>,
 'Self_Reflection.pdf': [<llama_index.core.tools.query_engine.QueryEngineTool at 0x22297c01120>,
 'loftq.pdf': [<llama_index.core.tools.query_engine.QueryEngineTool at 0x22296b8ffa0>,
 'swebench.pdf': [<llama_index.core.tools.query_engine.QueryEngineTool at 0x2228eb3b760>,
 'zipformer.pdf': [<llama_index.core.tools.query_engine.QueryEngineTool at 0x22295acc670>,
 'values.pdf': [<llama_index.core.tools.query_engine.QueryEngineTool at 0x2229509c0a0>,
 'finetune_fair_diffusion.pdf': [<llama_index.core.tools.query_engine.QueryEngineTool at 0x22296477c70>,
 'knowledge_card.pdf': [<llama_index.core.tools.query_engine.QueryEngineTool at 0x2229597ed40>,
 'metra.pdf': [<llama_index.core.tools.query_engine.QueryEngineTool at 0x2229768e0e0>,
 'vr_mcl.pdf': [<llama_index.core.tools.query_engine.QueryEngineTool at 0x222959

In [18]:
all_tools = [tool for document_tools in paper_to_tools_dict.values() for tool in document_tools]
all_tools

In [ ]:
# bad approach using similarity only
from llama_index.core import VectorStoreIndex
from llama_index.core.objects import ObjectIndex

obj_index = ObjectIndex.from_objects(
    all_tools,
    index_cls=VectorStoreIndex,
)
obj_retriever = obj_index.as_retriever(similarity_top_k=30)
tools = obj_retriever.retrieve(
    "Tell me about the eval dataset used in MetaGPT and SWE-Bench"
)
for tool in tools:
    print(tool.metadata.name, tool.metadata.description)

2026-05-09 06:42:25,541 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-09 06:42:26,806 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-09 06:42:27,136 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-09 06:42:27,358 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


summary_tool_inetune_fair_diffusion Useful for summarization questions related to inetune_fair_diffusion
vector_tool_inetune_fair_diffusion Useful for retrieving specific context from the inetune_fair_diffusion paper.
summary_tool_values Useful for summarization questions related to values
summary_tool_Self_Reflection Useful for summarization questions related to Self_Reflection
summary_tool_zipformer Useful for summarization questions related to zipformer
summary_tool_MetaGPT Useful for summarization questions related to MetaGPT
summary_tool_LongLoRA Useful for summarization questions related to LongLoRA
vector_tool_values Useful for retrieving specific context from the values paper.
summary_tool_swebench Useful for summarization questions related to swebench
summary_tool_knowledge_car Useful for summarization questions related to knowledge_car
vector_tool_Self_Reflection Useful for retrieving specific context from the Self_Reflection paper.
summary_tool_metra Useful for summarization

#### If very complex query (need multistep reasoning and tools in future)

* custom scoring: 0.7 * utilize_tool_now + 0.3 * utlize_tool_later

In [ ]:
async def score_tool(i, tool, query, llm):
    prompt = f"""Score this tool for the query (0-10):
Query: {query}
Tool: {tool.metadata.name} — {tool.metadata.description}
Respond ONLY with JSON: {{"now_score": 8, "later_score": 3, "reason": "..."}}"""
    
    response = await llm.acomplete(prompt)
    result = json.loads(response.text.strip().replace("```json","").replace("```",""))
    result["tool_index"] = i
    return result

async def llm_rerank_tools(query, tools, llm, top_k=3):
    scores = []
    for i, t in enumerate(tools):
        score = await score_tool(i, t, query, llm)
        scores.append(score)

    scores.sort(key = lambda x: x["now_score"] * 0.7 + x["later_score"] * 0.3, reverse=True)
    
    print("🔍 Tool ranking:")
    for s in scores[:top_k]:
        print(f"[{s['now_score']},{s['later_score']}] {tools[s['tool_index']].metadata.name} — {s['reason']}")
    
    return [tools[s["tool_index"]] for s in scores[:top_k]], scores

async def smart_tool_retriever(query: str, tools, llm, top_k=3):
    # can have obj_retriever layer before it (but it fails so won't add it)
    ranked_tools, scores = await llm_rerank_tools(query=query, tools=tools, llm=llm, top_k=top_k)
    return ranked_tools, scores

query = "Compare MetaGPT and LongLora approaches to training efficiency"
best_tools, scores = await llm_rerank_tools(query=query, tools=all_tools, llm=llm, top_k=3)
for tool in best_tools:
    print(tool.metadata.name)

2026-05-09 06:49:07,405 - INFO - AFC is enabled with max remote calls: 10.
2026-05-09 06:49:10,741 - INFO - AFC is enabled with max remote calls: 10.
2026-05-09 06:49:20,976 - INFO - AFC is enabled with max remote calls: 10.
2026-05-09 06:49:30,394 - INFO - AFC is enabled with max remote calls: 10.
2026-05-09 06:49:42,507 - INFO - AFC is enabled with max remote calls: 10.
2026-05-09 06:49:46,557 - INFO - AFC is enabled with max remote calls: 10.
2026-05-09 06:49:57,263 - INFO - AFC is enabled with max remote calls: 10.
2026-05-09 06:50:00,054 - INFO - AFC is enabled with max remote calls: 10.
2026-05-09 06:50:11,759 - INFO - AFC is enabled with max remote calls: 10.
2026-05-09 06:50:13,628 - INFO - AFC is enabled with max remote calls: 10.
2026-05-09 06:50:15,725 - INFO - AFC is enabled with max remote calls: 10.
2026-05-09 06:50:24,732 - INFO - AFC is enabled with max remote calls: 10.
2026-05-09 06:50:33,151 - INFO - AFC is enabled with max remote calls: 10.
2026-05-09 06:50:41,822 -

🔍 Tool ranking:
[8,3] vector_tool_MetaGPT — The tool is highly relevant for gathering technical details on MetaGPT to facilitate a comparison; however, it lacks information on LongLoRA, meaning a second tool or broader search would be required to complete the query.
[8,3] summary_tool_MetaGPT — The tool is highly relevant for the MetaGPT portion of the query, but it lacks the necessary information to provide a comparison or context on LongLoRA, making it insufficient for the complete task.
[5,2] vector_tool_LongLoRA — The tool is highly relevant for one half of the comparison (LongLoRA) but lacks any capability to retrieve or process information about MetaGPT, making it insufficient for a comprehensive comparative analysis.
vector_tool_MetaGPT
summary_tool_MetaGPT
vector_tool_LongLoRA


In [ ]:
# use selected tools for this query only
agent_workflow = AgentWorkflow.from_tools_or_functions(
    best_tools,
    llm=llm,
    system_prompt="Always use tools. Do not rely on prior knowledge."
)

memory = load_memory(path='./memory_manytools.json')
query = "Compare MetaGPT and LongLora approaches to training efficiency"
response = await agent_workflow.run(user_msg=query, memory=memory)
print(str(response))

2026-05-09 06:52:30,463 - INFO - AFC is enabled with max remote calls: 10.


To compare the training efficiency approaches of **LongLoRA** and **MetaGPT**, it is important to first distinguish their core objectives: **LongLoRA** is an *architectural optimization method* for fine-tuning LLMs, while **MetaGPT** is a *multi-agent framework* that organizes how models work together to complete complex tasks.

Here is a comparison of their approaches to efficiency:

### 1. The Core Efficiency Philosophy
*   **LongLoRA:** Focuses on **computational and memory efficiency**. Its goal is to minimize the resource cost (GPU memory and time) required to train a model to handle very long input sequences (up to 100k+ tokens). It achieves this by modifying the internal attention mechanism during the training phase.
*   **MetaGPT:** Focuses on **procedural and workflow efficiency**. Its goal is to maximize the success rate and quality of complex task completion (like software development) by using a multi-agent system. It treats "efficiency" as reducing the need for human inter

In [35]:
save_memory(memory, path='./memory_manytools.json')